In [1]:
import os
import pdfplumber
import pypdfium2 as pyfium
import chromadb
import time
from docx import Document
from pptx import Presentation
from IPython.display import display,Markdown
from PIL import Image
from dotenv import load_dotenv
from google import genai


In [2]:
load_dotenv()
client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [3]:
db_client = chromadb.PersistentClient(
    path= "./Chroma_db/"
)

In [4]:
try:
    collection = db_client.get_collection(
        name = "multimodal_rag"
    )
    print("collection loaded")
except:
    collection = db_client.create_collection(
        name="multimodal_rag"
    )
    print("collection created")

collection created


In [6]:
def extract_docx(file_path):
    doc = Document(file_path)
    text =""
    for paragraph in doc.paragraphs:
        if paragraph.text.strip():
            text += paragraph.text + "\n"
    return text

In [7]:
def get_processed_files(collection):
    results =collection.get()
    processed_files = set()
    for metadata in results["metadatas"]:
        processed_files.add(
            metadata["source"]
        )
    return processed_files

In [8]:
def load_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()
    documents = []
    if extension ==".txt":
        with open(file_path,"r",encoding="utf-8") as file:
            text =file.read()
        if text.strip():
            documents.append(
                {
                    "content":text,
                    "type":"text"
                }
            )
    
    elif extension ==".pdf":
        text =""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
            
                if page_text:
                    text += page_text+ "\n"
        text = text.replace("\u2028", "\n")
        text = text.replace("\u2029", "\n")
        if text.strip():
            documents.append(
                {
                    "content":text,
                    "type":"text"
                }
            )
    # elif extension == ".pptx":
    #     ppt_documents = extract_pptx(file_path)
    #     documents.extend(ppt_documents)
    elif extension == ".docx":
        text = extract_docx(file_path)
        documents.append(
            {
                "content":text,
                "type": "text"
            }
        )
    else:
        print(f"Skipping Unsupported file: {file_path} {extension}")
    return documents


In [9]:
def chunk_text(text,chunk_size=1000,overlap=200):
    chunks = []
    start =0
    while start<len(text):
        end = start+chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks    

In [10]:
def load_new_folder(folder_path,collection):
    processed_files= get_processed_files(collection)
    all_chunks = []
    for file in os.listdir(folder_path):
        if file.startswith("."):
            continue
        if file in processed_files:
            print(f"Skipping {file} (already embedded andstored)")
            continue
        file_path = os.path.join(
            folder_path,
            file
        )
        print(f"Reading file {file}")
        
        documents = load_document(file_path)
        for doc in documents:
            chunks = chunk_text(doc["content"])
            for i,chunk in enumerate(chunks):
                all_chunks.append(
                    {
                        "source":file,
                        "chunk_id": i,
                        "content": chunk,
                        "type":doc["type"]
                    }
                )
        print(f"Loaded{file}")
    return all_chunks

In [11]:
new_chunks = load_new_folder("../Data",collection)
print(
    f"Total Chunks: {len(new_chunks)}"
)

Reading file M3.pdf
LoadedM3.pdf
Reading file MLDOC.txt
LoadedMLDOC.txt
Total Chunks: 77


In [12]:
def create_embedding(text):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents = text
    )
    return response.embeddings[0].values

In [13]:

for chunk in new_chunks:
    embedding = create_embedding(chunk["content"])
    collection.add(
        ids = [f"{chunk['source']}_{chunk['chunk_id']}"],
        documents = [chunk["content"]],
        embeddings = [embedding],
        metadatas=[
            {
                "source" : chunk["source"],
                "chunk_id" :chunk["chunk_id"],
                "type": chunk["type"]
            }
        ]
       )
print("stored successfully")


stored successfully


In [14]:
def retrieve_chunks(query):
    if not query.strip():
        return None
    query_embedding = create_embedding(query)
    results = collection.query(
        query_embeddings =[query_embedding],
        n_results = 3
    )
    return results

In [28]:
results = retrieve_chunks(
    "What is supervised machine learning ?"
)

print(results)

{'ids': [['MLDOC.txt_7', 'MLDOC.txt_9', 'MLDOC.txt_6']], 'embeddings': None, 'documents': [[' machine learning\nSupervised learning, also known as supervised machine learning, is defined by its use\nof labeled datasets to train algorithms to classify data or predict outcomes accurately.\nAs input data is fed into the model, the model adjusts its weights until it has been fitted\nappropriately. This occurs as part of the cross validation process to ensure that the\nmodel avoids overfitting or underfitting. Supervised learning helps organizations solve a\nvariety of real-world problems at scale, such as classifying spam in a separate folder\nfrom your inbox. Some methods used in supervised learning include neural networks,\nnaïve bayes, linear regression, logistic regression, random forest, and support vector\nmachine (SVM).\nUnsupervised machine learning\nUnsupervised learning, also known as unsupervised machine learning, uses machine\nlearning algorithms to analyze and cluster unlabele

In [32]:
def build_context(results):
    context = ""
    for doc in results["documents"][0]:
        context += doc
        context += "\n\n"
    return context

In [21]:
def classify_query(query):

   query = query.lower()

   academic_keywords = [

      "assignment",
      "mcq",
      "summary",
      "notes",
      "exam",
      "question",
      "important question",
      "interview",
      "placement",
      "career",
      "roadmap",
      "project",
      "python",
      "java",
      "sql",
      "javascript",
      "machine learning",
      "deep learning",
      "ai",
      "artificial intelligence",
      "rag",
      "llm",
      "agent",
      "dsa",
      "data structure",
      "algorithm",
      "resume",
      "operating system",
      "dbms",
      "computer network",
      "aptitude",
      "coding"
   ]

   blocked_keywords = [
      "who is",
      "stock",
      "share market",
      "crypto",
      "bitcoin",
      "politics",
      "election",
      "prime minister",
      "president",
      "movie",
      "actor",
      "actress",
      "celebrity",
      "ipl",
      "football",
      "cricket",
      "weather",
      "news"
   ]

   for keyword in blocked_keywords:

      if keyword in query:
         return "non-academic"

   # for keyword in academic_keywords:

   #    if keyword in query:
   #       return "academic"

   return "academic"

In [23]:
print(classify_query("who is virat kohli"))

non-academic


In [25]:
def detect_intent(question):
    question = question.lower()
    if "mcq" in question or "multiple choice questions" in question:
        return "mcq"
    elif "summary" in question or "summarize" in question:
        return "summary"
    elif "imp question" in question or "important question" in question:
        return "important_question"
    elif "solve" in question or "assignment" in question:
        return "assignment"
    elif "interview" in question or "viva" in question:
        return "interview_prep"
    elif "placement" in question or "off-campus" in question:
        return "placement_prep"
    elif "roadmap" in question or "career" in question :
        return "career_guidance"
    elif "project idea" in question or "project ideas" in question or "project" in question :
        return "project_idea"
    else:
        return "explanation"

In [26]:
print(detect_intent("give a 10 MCQ's on supervised"))

mcq


In [27]:
def create_prompt(intent, context, question,retrieval_found):
    if retrieval_found:

        source_instruction = f"""
        Use the provided study material as the PRIMARY source.

        If the answer exists in the study material,
        answer using the study material.

        Do not ignore the provided context.

        Study Material:
        {context}
        """

    else:

        source_instruction = """
        No relevant study material was found.

        Answer using your academic knowledge.

        You may answer ONLY questions related to:

        - Academic subjects
        - Assignments
        - Exams
        - Interview preparation
        - Placement preparation
        - Career guidance
        - Programming
        - Artificial Intelligence
        - Machine Learning
        - Data Structures & Algorithms
        - Project ideas

        Do NOT answer:

        - Politics
        - News
        - Stock market
        - Cryptocurrency
        - Sports
        - Entertainment
        - Celebrity topics
        """
    if intent == "mcq":

        return f"""
            You are an AI Academic Assistant.

            Using ONLY the provided context,
            generate 10 MCQs.

            For each MCQ provide:

            1. Question
            2. Four Options
            3. Correct Answer
            4. Explanation

            Context:
            {source_instruction}

            Student Request:
            {question}
            """

    elif intent == "summary":

        return f"""
            You are an AI Academic Assistant.

            Create concise exam notes.

            Include:

            - Key Concepts
            - Important Definitions
            - Important Points

            Context:
            {source_instruction}

            Student Request:
            {question}
            """

    elif intent == "important_questions":

        return f"""
            You are an AI Academic Assistant.

            Generate likely university exam questions.

            Provide:

            - 2 Marks Questions
            - 5 Marks Questions
            - 10 Marks Questions

            Context:
            {source_instruction}

            Student Request:
            {question}
            """

    elif intent == "assignment":

        return f"""
            You are an AI Academic Assistant.

            Solve the assignment question step by step.

            Use only the provided context.

            Context:
            {source_instruction}

            Student Request:
            {question}
            """
    elif intent == "interview_prep":

        return f"""
        You are an AI Academic Assistant.

        Help the student prepare for interviews.

        Include:

        - Important concepts to study
        - Common interview questions
        - Preparation strategy
        - Mistakes to avoid

        Student Question:
        {question}
        """
    elif intent == "placement_prep":

        return f"""
        You are an AI Academic Assistant.

        Help the student prepare for placements.

        Include:

        - Preparation roadmap
        - Important skills
        - Aptitude preparation
        - Technical preparation
        - Interview preparation tips

        Student Question:
        {question}
        """
    elif intent == "career_guidance":

        return f"""
        You are an AI Academic Assistant.

        Provide a structured career roadmap.

        Include:

        - Skills to learn
        - Technologies to focus on
        - Projects to build
        - Resources
        - Career opportunities

        Student Question:
        {question}
        """
    elif intent == "project_idea":

        return f"""
        You are an AI Academic Assistant.

        Suggest practical project ideas.

        For each project provide:

        - Project Title
        - Difficulty Level
        - Tech Stack
        - Features
        - Learning Outcomes

        Student Question:
        {question}
        """
    else:

        return f"""
            You are an AI Academic Assistant.

            Explain clearly in simple student-friendly language.

            Context:
            {source_instruction}

            Student Question:
            {question}
            """

In [29]:
SIMILARITY_THRESHOLD = 1.2

def validate_retrieval(results):

    if not results:
        return False

    documents = results.get("documents", [])
    distances = results.get("distances", [])

    if not documents or not documents[0]:
        return False

    if not distances or not distances[0]:
        return False

    best_distance = distances[0][0]

    return best_distance <= SIMILARITY_THRESHOLD

In [42]:
def ask_question(query):
    query_type = classify_query(query)
    if query_type =="non-academic":
        return {
        "answer":
        """
        This assistant is designed only for
        academic learning, placement preparation,
        interview preparation and career guidance.

        Please ask an academic-related question.
        """,
        "sources":[]
        }
    results = retrieve_chunks(query)
    # print(results)
    retrieval_found = validate_retrieval(results)
    if retrieval_found:
        context = build_context(results)
        metadata = results["metadatas"][0]
    else:
        context = ""
        metadata = []
    intent = detect_intent(query)
    # Step 5: Create prompt
    prompt = create_prompt(
        intent,context,query,retrieval_found
    ) 
     

    # Step 6: Generate answer
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    sources = []
    for item in metadata:
        sources.append(
            f"{item['source']} (Chunk {item['chunk_id']})"
        )
    return {
        "answer":response.text,
        "sources": sources
        }
    

In [43]:
while True:
    query = input("\nEnter a Question:")
    if query.lower()== "exit":
        break
    Answer = ask_question(query)
    display(Markdown(Answer["answer"]))
    print("\nSources:")

    for source in Answer["sources"]:
        print("-", source)

That's a fantastic question! Preparing for a Data Analytics role interview requires a blend of technical skills, strong problem-solving abilities, and effective communication. As your AI Academic Assistant, I'll help you break down what you need to study, what questions to expect, how to strategize, and common pitfalls to avoid.

---

## Preparing for a Data Analytics Role Interview

A data analyst acts as a bridge between raw data and business decision-making. Interviewers will be looking for your ability to extract insights, tell data-driven stories, and impact business outcomes.

### 1. Important Concepts to Study

These are the pillars of data analytics. Ensure you have a solid understanding and can speak to their practical application.

#### A. Core Technical Skills

1.  **SQL (Structured Query Language):** This is often the *most critical* technical skill.
    *   **Fundamentals:** SELECT, FROM, WHERE, GROUP BY, ORDER BY, LIMIT.
    *   **Joins:** INNER, LEFT, RIGHT, FULL OUTER joins – understand when and why to use each.
    *   **Subqueries & CTEs (Common Table Expressions):** For complex queries.
    *   **Window Functions:** RANK(), ROW_NUMBER(), LEAD(), LAG(), NTILE() – very common for ranking and comparisons.
    *   **Aggregations:** COUNT, SUM, AVG, MIN, MAX.
    *   **Data Manipulation:** INSERT, UPDATE, DELETE (basic understanding).
    *   **Date & String Functions:** EXTRACT, DATE_DIFF, SUBSTRING, CONCAT.
2.  **Programming Languages (Python or R):** Depending on the role, one or both might be required.
    *   **Data Manipulation (Python/Pandas, R/dplyr):** Loading data, cleaning (handling missing values, outliers, data types), filtering, transforming, merging datasets.
    *   **Data Visualization (Python/Matplotlib, Seaborn; R/ggplot2):** Creating various plots (scatter, line, bar, histogram, box plots) and understanding their appropriate use.
    *   **Basic Statistics:** Applying statistical functions (mean, median, mode, standard deviation, correlation).
    *   **Basic Machine Learning (Optional but a plus for some roles):** Understanding concepts of regression, classification, clustering, and how to apply simple models (e.g., linear regression, logistic regression).
3.  **Spreadsheet Software (Excel/Google Sheets):**
    *   **Formulas:** VLOOKUP/XLOOKUP, INDEX/MATCH, SUMIF/COUNTIF, Pivot Tables, Conditional Formatting.
    *   **Data Cleaning:** Text-to-columns, removing duplicates.
    *   **Charting:** Creating effective charts.
4.  **Data Visualization & Business Intelligence (BI) Tools:**
    *   **Concepts:** Principles of good data visualization, choosing the right chart type, storytelling with data.
    *   **Tools (Tableau, Power BI, Looker, Qlik Sense):** Experience building interactive dashboards, understanding data connections, creating calculated fields.
5.  **Statistics & Probability:**
    *   **Descriptive Statistics:** Mean, median, mode, variance, standard deviation, quartiles, percentiles.
    *   **Inferential Statistics:** Hypothesis testing (t-tests, chi-squared tests, ANOVA), p-values, confidence intervals.
    *   **Distributions:** Normal distribution, skewed distributions.
    *   **A/B Testing:** Design, interpretation, potential pitfalls.
    *   **Correlation vs. Causation:** Understanding the crucial difference.

#### B. Foundational Concepts & Methodologies

1.  **Data Life Cycle:** Understanding the stages: data collection, storage, cleaning, analysis, interpretation, visualization, and deployment.
2.  **ETL/ELT:** Basic understanding of Extract, Transform, Load processes – how data moves and is prepared.
3.  **Problem Solving & Critical Thinking:** How to approach an ambiguous problem, break it down, and use data to solve it.
4.  **Business Acumen:** Understanding the business context, key performance indicators (KPIs), and how data insights drive business decisions.
5.  **Data Quality & Governance:** Importance of clean data, data integrity, and ethical considerations.

#### C. Soft Skills

1.  **Communication & Storytelling:** Explaining complex technical concepts to non-technical stakeholders, presenting findings clearly and persuasively.
2.  **Collaboration:** Working effectively with engineers, product managers, and other teams.
3.  **Curiosity & Proactiveness:** A desire to dig deeper and find answers.

---

### 2. Common Interview Questions

Interviews usually consist of a mix of behavioral, technical, and case study questions.

#### A. Behavioral Questions (Use the STAR Method: Situation, Task, Action, Result)

1.  "Tell me about yourself." (Craft a concise narrative connecting your past to this role).
2.  "Why data analytics? Why this company/role?"
3.  "Describe a challenging data project you worked on. What was your role, what challenges did you face, and what was the outcome?"
4.  "Tell me about a time you had to explain a complex technical concept to a non-technical audience."
5.  "How do you handle disagreement or conflicting opinions with team members or stakeholders?"
6.  "Describe a time you made a mistake in your analysis. How did you identify it, and what did you do?"
7.  "How do you prioritize your work when you have multiple competing tasks?"
8.  "What are your strengths and weaknesses?"
9.  "Where do you see yourself in 3-5 years?"

#### B. Technical Questions

1.  **SQL:**
    *   "Write a SQL query to find the top 5 customers by total order value."
    *   "Given two tables, `users` and `orders`, write a query to find users who have placed more than 3 orders."
    *   "Explain the difference between `LEFT JOIN` and `INNER JOIN`."
    *   "What are window functions, and when would you use them?"
2.  **Python/R:**
    *   "How would you clean a dataset with missing values and outliers?"
    *   "Write a Python function to calculate the average of a list of numbers, ignoring non-numeric values."
    *   "Explain what a Pandas DataFrame is."
    *   "When would you choose a histogram over a bar chart?"
3.  **Statistics & A/B Testing:**
    *   "Explain p-value in simple terms."
    *   "What is the difference between correlation and causation? Give an example."
    *   "How would you design an A/B test for a new website feature?"
    *   "What are some potential pitfalls of A/B testing?"
4.  **Data Visualization:**
    *   "What makes a good data visualization? What are your favorite chart types?"
    *   "How would you visualize sales data across different regions over time?"

#### C. Case Study / Business Acumen Questions

1.  "Our user engagement has dropped by 10% this week. How would you investigate this, and what metrics would you look at?"
2.  "A product manager wants to launch a new feature that slightly increases revenue but decreases user satisfaction. How would you advise them using data?"
3.  "Given this dataset [they might provide a small sample or describe one], what insights can you find? What business recommendations would you make?"
4.  "What are the key metrics you would track for an e-commerce website/social media app/SaaS product?"

#### D. Questions for the Interviewer

Always have a few thoughtful questions prepared to ask them. This shows engagement and intellectual curiosity.
*   "What does a typical day look like for a data analyst on this team?"
*   "What are some of the biggest data challenges facing the company/team right now?"
*   "How does the data analytics team collaborate with other departments?"
*   "What opportunities are there for professional development and growth within this role?"

---

### 3. Preparation Strategy

#### A. Understand the Role and Company

1.  **Job Description Deep Dive:** Highlight keywords, required skills, and responsibilities. Tailor your answers and resume to match.
2.  **Company Research:** Understand their products/services, recent news, mission, values, and industry position. Look for how they use data.
3.  **Glassdoor/LinkedIn:** Look up common interview questions for that company/role, and research interviewers' backgrounds.

#### B. Technical Skill Practice

1.  **SQL:**
    *   **LeetCode/HackerRank/StrataScratch:** Practice SQL problems daily. Focus on medium-hard problems for joins, subqueries, and window functions.
    *   **Build a database (even a simple one):** Create tables, insert data, and practice queries to reinforce concepts.
2.  **Python/R:**
    *   **Kaggle:** Work through data cleaning and analysis notebooks. Participate in beginner competitions.
    *   **DataCamp/Codecademy:** Review syntax and data manipulation libraries (Pandas/dplyr).
    *   **Personal Projects:** Build a small portfolio project from scratch (e.g., analyze a public dataset, build a simple dashboard).
3.  **Statistics:**
    *   **Review your notes/textbooks:** Re-familiarize yourself with core concepts.
    *   **Online courses:** (e.g., Coursera, Khan Academy) for refreshers.
    *   **Apply concepts:** Think about how you'd use them in real-world scenarios.
4.  **BI Tools:**
    *   **Create a sample dashboard:** Download Tableau Public or Power BI Desktop and connect to a public dataset (like Superstore for Tableau) to build a dashboard. Practice creating calculated fields and different chart types.

#### C. Behavioral Interview Preparation

1.  **Identify Key Experiences:** List 5-7 significant experiences from your past projects, work, or academics where you demonstrated problem-solving, teamwork, leadership, overcoming challenges, etc.
2.  **STAR Method:** For each experience, outline the Situation, Task, Action, and Result. Practice telling these stories concisely and effectively.
3.  **Practice "Tell me about yourself":** Craft a compelling 60-90 second elevator pitch that highlights your relevant skills, experiences, and enthusiasm for the role.
4.  **Mock Interviews:** Practice with a friend, mentor, or career services. Get feedback on your clarity, confidence, and how well you answer questions.

#### D. Portfolio & Projects

1.  **Curate your portfolio:** Select 2-3 of your strongest, most relevant data analytics projects.
2.  **Be ready to explain them deeply:** For each project, articulate:
    *   The problem you were trying to solve.
    *   The data sources and tools you used.
    *   Your specific role and contributions.
    *   The challenges you faced and how you overcame them.
    *   The key insights you found.
    *   The impact or recommendations you made.
    *   What you would do differently next time.
3.  **Shareable Link:** Have a link to your GitHub or a personal website with your projects ready.

#### E. Communication & Storytelling

1.  **Simplify complex ideas:** Practice explaining technical terms to someone without a data background.
2.  **Structure your answers:** For case studies, think aloud, articulate your thought process (clarifying questions, hypothesis, data needed, analysis steps, potential insights, recommendations).
3.  **Focus on impact:** Always connect your analysis back to business value.

---

### 4. Mistakes to Avoid

1.  **Lack of Research:** Not knowing anything about the company, its products, or recent news. This shows a lack of genuine interest.
2.  **Weak SQL Fundamentals:** Many candidates underestimate SQL; it's a common gatekeeper. Don't just know syntax, understand optimization and different join types thoroughly.
3.  **Poor Communication:** Rambling, using too much jargon, or being unable to explain findings clearly. Data analysts need to translate data into actionable insights for non-technical people.
4.  **Not Explaining Your Projects Effectively:** Don't just list tools. Focus on the "why," the "how," and the "impact" of your projects. What problem did you solve? What was the result?
5.  **Forgetting to Ask Questions:** This indicates disinterest. Always have thoughtful questions prepared.
6.  **Guessing or Bluffing:** If you don't know an answer, it's better to admit it honestly. You can say, "That's a good question, I'm not entirely sure, but based on my understanding, I would approach it by..." or "I haven't encountered that specifically, but I would look into X or Y."
7.  **Ignoring Behavioral Questions:** These are just as important as technical questions. They assess your fit with the team and company culture.
8.  **Over-reliance on Theory:** Interviewers want to see how you *apply* your knowledge, not just if you can recite definitions. Show practical examples.
9.  **Negative Attitude:** Never speak negatively about past employers, colleagues, or projects.
10. **Not Following Up:** Always send a thank-you email within 24 hours of the interview, reiterating your interest and perhaps mentioning something specific from your conversation.

---

Remember, every interview is a learning experience. Even if you don't get the first role, the preparation and practice will make you stronger for the next one. Good luck!


Sources:
- M3.pdf (Chunk 9)
- M3.pdf (Chunk 40)
- M3.pdf (Chunk 11)


Hello! I'm ready when you are. Please provide your student question, and I'll do my best to explain it clearly in simple, student-friendly language, based on my academic knowledge.


Sources:


Hello! I'm here to help you out with your academic questions.

It sounds like you might be having trouble finding study material, but don't worry, I can use my knowledge to explain things to you.

Please go ahead and ask your question related to any of the following topics:

*   Academic subjects
*   Assignments
*   Exams
*   Interview preparation
*   Placement preparation
*   Career guidance
*   Programming
*   Artificial Intelligence
*   Machine Learning
*   Data Structures & Algorithms
*   Project ideas

I'm ready when you are!


Sources:


I understand you're having trouble finding study material. Don't worry, I'm here to help!

Please go ahead and ask your question. I'll do my best to explain it clearly in simple, student-friendly language, drawing from my academic knowledge.

Remember, I can help with questions related to:
*   Academic subjects
*   Assignments
*   Exams
*   Interview preparation
*   Placement preparation
*   Career guidance
*   Programming
*   Artificial Intelligence
*   Machine Learning
*   Data Structures & Algorithms
*   Project ideas


Sources:


KeyboardInterrupt: 